<a href="https://colab.research.google.com/github/chetools/CHE4071_Spring2026/blob/main/pH_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [202]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [297]:
def netH(pH):
    H = 10**-pH
    OH = 1e-14 / H
    netH = H - OH
    return netH
    # return jnp.where(pH<7, 10**(-pH), -10**(-14+pH))

def pH(netH):
    #x = 1e-14/(netH + x), x=H+ and OH- from H2O hydrolysis and charge balance
    #x**2 + netH*x -1e-14 = 0
    d= jnp.sqrt(netH**2 + 4*1e-14)
    x = jnp.where(netH>0, (-netH + d)/2, (-netH - d)/2)
    return jnp.squeeze(jnp.where(netH>0, -jnp.log10(netH+x), 14.+jnp.log10(-netH-x)))

dpH_from_netH = jax.grad(pH)

In [298]:
def make_step(t0):
    def step(t):
        return jnp.where(t>t0, 1, 0.)
    return step

In [299]:
V = 10.  #L
pHA, pHB = 1., 14.
Kc = 3e2

In [300]:
def dpHdt(t, pH, u):
    qin, pHin = u
    netH_err = netH(pH_setpoint)-netH(pH)
    qA = jnp.where(netH_err>0, Kc*jnp.abs(netH_err), 0.)
    qB = jnp.where(netH_err<0, Kc*jnp.abs(netH_err), 0.)

    dnetH_dt = (qin*netH(pHin) + qA*netH(pHA) + qB*netH(pHB) - (qin+qA+qB)*netH(pH))/V
    return dpH_netH(pH) * dnetH_dt

In [301]:
def steps(t, s, t0, w):
    return jnp.sum(s*jax.scipy.special.expit((t-t0)/w))

In [302]:
V=10. #L
pH_init= 7.
qcA_pH = 1
qcB_pH = 14
pH_setpoint = 7.
c_setpoint = netH(pH_setpoint)
Kc = 3e2
taui = 1e10

def rhs(t, vec):
    pH, eint = vec
    qin = 1.
    qin_pH = 7. + steps(t, jnp.array([1, -4., 4.]), jnp.array([1, 20, 50]), 0.1)

    cH = netH(pH)
    cerr = c_setpoint - cH
    q_control = Kc*(cerr + eint/taui)
    qcA = jnp.where(q_control>=0, q_control, 0.)
    qcB = jnp.where(q_control<0, -q_control, 0.)

    dnetHdt = (qin*netH(qin_pH) + qcA*netH(qcA_pH) + qcB*netH(qcB_pH) - (qin+qcA+qcB)*cH)/V
    dpHdt = dpH_from_netH(cH)*dnetHdt
    return jnp.array([dpHdt, cerr])

rhs_jit = jax.jit(rhs)
rhs_jac = jax.jit(jax.jacobian(rhs, argnums=1))

dt=0.001
def step(state, x):
    t, vec = state
    new_vec = rhs_jit(t, vec)*dt + vec
    return (t+dt, new_vec), (t, new_vec)

_, (ts, vecs) = jax.lax.scan(step, (0, jnp.array([pH_init,0.])), None, length=100000)
fig=make_subplots()
fig.add_scatter(x=ts, y=vecs[:, 0], mode='lines')
fig.update_yaxes(range=[4,10])